[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoseGG8/AutoEncoderPi/blob/main/ETL_inicial/ETL_inicial.ipynb)

Se ejecuta en COLAB

In [ ]:
import os
import sys
import shutil
import urllib.request
import subprocess
import scipy.io
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 0. Configuración Automática de Dependencias en Google Colab
# ---------------------------------------------------------
print("[1/5] Preparando entorno en Google Colab...")
try:
    import rarfile
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rarfile"])
    import rarfile

if sys.platform.startswith('linux'):
    if not shutil.which("unrar"):
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=False)

# ---------------------------------------------------------
# 1. Rutas de almacenamiento DENTRO DE COLAB (/content)
# ---------------------------------------------------------
colab_dir = "/content"
csv_output_path = os.path.join(colab_dir, 'paderborn_64khz_processed.csv')
temp_dir = os.path.join(colab_dir, 'temp_paderborn_downloads')
os.makedirs(temp_dir, exist_ok=True)

# ---------------------------------------------------------
# 2. Catálogo completo de Zenodo y LISTA CONFIGURABLE
# ---------------------------------------------------------
ZENODO_CATALOG = {
    # --- MOTORES SANOS (NORMALES) ---
    "K001.rar": "https://zenodo.org/records/15845309/files/K001.rar?download=1",
    "K002.rar": "https://zenodo.org/records/15845309/files/K002.rar?download=1",
    "K003.rar": "https://zenodo.org/records/15845309/files/K003.rar?download=1",
    "K004.rar": "https://zenodo.org/records/15845309/files/K004.rar?download=1",
    "K005.rar": "https://zenodo.org/records/15845309/files/K005.rar?download=1",
    "K006.rar": "https://zenodo.org/records/15845309/files/K006.rar?download=1",

    # --- FALLAS ARTIFICIALES (EDM) ---
    "KA01.rar": "https://zenodo.org/records/15845309/files/KA01.rar?download=1",
    "KA03.rar": "https://zenodo.org/records/15845309/files/KA03.rar?download=1",
    "KA05.rar": "https://zenodo.org/records/15845309/files/KA05.rar?download=1",
    "KA07.rar": "https://zenodo.org/records/15845309/files/KA07.rar?download=1",
    "KA08.rar": "https://zenodo.org/records/15845309/files/KA08.rar?download=1",
    "KA09.rar": "https://zenodo.org/records/15845309/files/KA09.rar?download=1",
    "KI01.rar": "https://zenodo.org/records/15845309/files/KI01.rar?download=1",
    "KI03.rar": "https://zenodo.org/records/15845309/files/KI03.rar?download=1",
    "KI05.rar": "https://zenodo.org/records/15845309/files/KI05.rar?download=1",
    "KI07.rar": "https://zenodo.org/records/15845309/files/KI07.rar?download=1",
    "KI08.rar": "https://zenodo.org/records/15845309/files/KI08.rar?download=1",
    "KI09.rar": "https://zenodo.org/records/15845309/files/KI09.rar?download=1",

    # --- FALLAS REALES (FATIGA / VIDA ACELERADA) ---
    "KA04.rar": "https://zenodo.org/records/15845309/files/KA04.rar?download=1",
    "KA15.rar": "https://zenodo.org/records/15845309/files/KA15.rar?download=1",
    "KA16.rar": "https://zenodo.org/records/15845309/files/KA16.rar?download=1",
    "KA22.rar": "https://zenodo.org/records/15845309/files/KA22.rar?download=1",
    "KA30.rar": "https://zenodo.org/records/15845309/files/KA30.rar?download=1",
    "KI04.rar": "https://zenodo.org/records/15845309/files/KI04.rar?download=1",
    "KI14.rar": "https://zenodo.org/records/15845309/files/KI14.rar?download=1",
    "KI16.rar": "https://zenodo.org/records/15845309/files/KI16.rar?download=1",
    "KI18.rar": "https://zenodo.org/records/15845309/files/KI18.rar?download=1",
    "KI21.rar": "https://zenodo.org/records/15845309/files/KI21.rar?download=1",
    "KB23.rar": "https://zenodo.org/records/15845309/files/KB23.rar?download=1",
    "KB24.rar": "https://zenodo.org/records/15845309/files/KB24.rar?download=1",
    "KB27.rar": "https://zenodo.org/records/15845309/files/KB27.rar?download=1"
}

# =========================================================================
# LISTA CONFIGURABLE DE ARCHIVOS A DESCARGAR
# (Modifica esta lista agregando o quitando nombres según lo que necesites)
# =========================================================================
ARCHIVOS_A_DESCARGAR = [
    "K001.rar",
    "K002.rar",
    "KA01.rar",
    "KI01.rar",
    "KB23.rar"
]

# ---------------------------------------------------------
# 3. Funciones de Descarga y Extracción en Linux / Colab
# ---------------------------------------------------------
def download_and_extract_selected(file_list, catalog, target_folder):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    
    print(f"\n[2/5] Descargando {len(file_list)} archivos en la memoria local de Colab ({target_folder})...")
    
    for filename in file_list:
        if filename not in catalog:
            print(f"  [!] Advertencia: '{filename}' no está en el catálogo. Omitiendo.")
            continue
            
        url = catalog[filename]
        rar_path = os.path.join(target_folder, filename)
        
        # Descarga si no existe en Colab
        if not os.path.exists(rar_path):
            print(f"\nIniciando descarga de {filename} desde Zenodo...")
            req = urllib.request.Request(url, headers=headers)
            try:
                with urllib.request.urlopen(req) as response, open(rar_path, 'wb') as out_file:
                    total_size = int(response.headers.get('Content-Length', 0))
                    block_size = 1024 * 1024
                    downloaded = 0
                    
                    while True:
                        buffer = response.read(block_size)
                        if not buffer:
                            break
                        downloaded += len(buffer)
                        out_file.write(buffer)
                        
                        if total_size > 0:
                            percent = (downloaded / total_size) * 100
                            mb_down = downloaded / (1024 * 1024)
                            mb_total = total_size / (1024 * 1024)
                            sys.stdout.write(f"\r  [⬇] Progreso Colab: {mb_down:.1f} MB / {mb_total:.1f} MB ({percent:.1f}%)")
                        else:
                            mb_down = downloaded / (1024 * 1024)
                            sys.stdout.write(f"\r  [⬇] Descargado: {mb_down:.1f} MB")
                        sys.stdout.flush()
                print(f"\n  [✓] Descarga completada: {filename}")
            except Exception as e:
                print(f"\n  [X] Error descargando {filename}: {e}")
                if os.path.exists(rar_path):
                    os.remove(rar_path)
                continue
        else:
            print(f"\n[✓] El archivo {filename} ya existe en el disco temporal de Colab.")
            
        # Descompresión nativa en Linux
        print(f"  Descomprimiendo {filename}...")
        try:
            res = subprocess.run(["unrar", "x", "-o+", rar_path, target_folder], capture_output=True, text=True)
            if res.returncode == 0:
                print("  [✓] Descomprimido correctamente.")
            else:
                rf = rarfile.RarFile(rar_path)
                rf.extractall(path=target_folder)
                print("  [✓] Descomprimido con rarfile.")
        except Exception as e:
            print(f"  [X] Error extrayendo {filename}: {e}. Se eliminará para re-descargarlo.")
            if os.path.exists(rar_path):
                os.remove(rar_path)

# ---------------------------------------------------------
# 4. Métodos ETL (64 kHz)
# ---------------------------------------------------------
def parse_metadata_from_path(file_path):
    path_upper = file_path.upper()
    rpm = 900 if "N09" in path_upper else 1500
    torque = 0.1 if "M01" in path_upper else 0.7
    force = 400 if "F04" in path_upper else 1000
    
    if "KA" in path_upper:
        fault_type, label = "Pista_Exterior", 1
    elif "KI" in path_upper:
        fault_type, label = "Pista_Interior", 1
    elif "KB" in path_upper:
        fault_type, label = "Combinado", 1
    elif "K0" in path_upper:
        fault_type, label = "Normal", 0
    else:
        fault_type, label = "Anomalia", 1
        
    return rpm, torque, force, fault_type, label

def extraer_vector_1d(data_obj):
    arr = data_obj
    while isinstance(arr, np.ndarray) and arr.dtype == object and arr.size > 0:
        arr = arr.flat[0]
    return np.ravel(arr)

def get_64khz_channels(mat_path):
    mat = scipy.io.loadmat(mat_path)
    key = [k for k in mat.keys() if not k.startswith('__')][0]
    data_struct = mat[key][0, 0]
    
    y_channels = data_struct['Y'] 
    n_channels = y_channels.shape[1] if y_channels.ndim > 1 else y_channels.shape[0]
    
    vib, i1, i2 = None, None, None
    for i in range(n_channels):
        ch = y_channels[0, i] if y_channels.ndim > 1 else y_channels[i]
        ch_name = ''
        if 'Name' in ch.dtype.names:
            name_val = ch['Name']
            while isinstance(name_val, np.ndarray) and name_val.size > 0:
                name_val = name_val.flat[0]
            ch_name = str(name_val).lower()
            
        signal = extraer_vector_1d(ch['Data'])
        if len(signal) == 256000:
            if 'vib' in ch_name or 'acc' in ch_name:
                vib = signal
            elif 'phase_current_1' in ch_name or 'current_1' in ch_name:
                i1 = signal
            elif 'phase_current_2' in ch_name or 'current_2' in ch_name:
                i2 = signal

    return vib, i1, i2

def process_all_64khz_mats(input_dir, window_size=2048):
    all_records = []
    mat_files = []
    
    for root, _, files in os.walk(input_dir):
        for f in files:
            if f.lower().endswith('.mat') and not f.startswith('._'):
                mat_files.append(os.path.join(root, f))
                
    print(f"\n[3/5] Procesando {len(mat_files)} archivos .mat desempacados...")
    
    for mat_path in mat_files:
        filename = os.path.basename(mat_path)
        rpm, torque, force, fault_type, label = parse_metadata_from_path(mat_path)
        
        try:
            vib, i1, i2 = get_64khz_channels(mat_path)
            if vib is None:
                continue
                
            num_windows = len(vib) // window_size
            for i in range(num_windows):
                idx_start = i * window_size
                idx_end = (i + 1) * window_size
                seg_vib = vib[idx_start:idx_end]
                
                rms_vib = np.sqrt(np.mean(seg_vib**2))
                kurt_vib = np.mean(((seg_vib - np.mean(seg_vib)) / (np.std(seg_vib) + 1e-8))**4)
                peak_vib = np.max(np.abs(seg_vib))
                
                record = {
                    'RPM': rpm,
                    'Torque_Nm': torque,
                    'Fuerza_N': force,
                    'RMS_Vib': rms_vib,
                    'Kurtosis_Vib': kurt_vib,
                    'Peak_Vib': peak_vib,
                    'Tipo_Falla': fault_type,
                    'Label': label
                }
                
                if i1 is not None:
                    seg_i1 = i1[idx_start:idx_end]
                    record['RMS_I1'] = np.sqrt(np.mean(seg_i1**2))
                if i2 is not None:
                    seg_i2 = i2[idx_start:idx_end]
                    record['RMS_I2'] = np.sqrt(np.mean(seg_i2**2))
                    
                all_records.append(record)
                
            print(f"  [✓] Ventanas extraídas: {filename} ({num_windows} ventanas | Estado: {fault_type})")
            
        except Exception as e:
            print(f"  [X] Error procesando {filename}: {e}")
            
    return pd.DataFrame(all_records)

# ---------------------------------------------------------
# 5. Flujo Principal de Ejecución en Colab
# ---------------------------------------------------------
if __name__ == "__main__":
    print("==================================================")
    print(" PIPELINE AUTOMÁTICO PADERBORN 64 kHz (GOOGLE COLAB) ")
    print("==================================================")
    
    # 1. Descarga y extracción en disco de Colab
    download_and_extract_selected(ARCHIVOS_A_DESCARGAR, ZENODO_CATALOG, temp_dir)
    
    # 2. ETL a 64 kHz
    df_result = process_all_64khz_mats(temp_dir)
    
    # 3. Guardar CSV en el directorio base de Colab (/content)
    print("\n[4/5] Guardando resultados...")
    if len(df_result) > 0:
        df_result.to_csv(csv_output_path, index=False)
        print("\n==================================================")
        print(f" [5/5] CSV GUARDADO EXITOSAMENTE EN COLAB ")
        print(f" Ruta en Colab: {csv_output_path}")
        print(f" Total de ventanas procesadas: {len(df_result)}")
        print("\n Resumen de clases en el CSV:")
        print(df_result['Tipo_Falla'].value_counts())
        print("==================================================")
    else:
        print("\n[!] No se pudieron generar datos. Verifica la lista de archivos seleccionados.")

 PIPELINE AUTOMÁTICO DE PADERBORN (64 kHz) 

Descargando K001.rar desde Zenodo...
  [⬇] Progreso: 165.8 MB / 165.8 MB (100.0%)
  [✓] Descarga completada: K001.rar
  Descomprimiendo K001.rar...
  [✓] Descomprimido exitosamente mediante 'tar' del sistema.

Descargando K002.rar desde Zenodo...
  [⬇] Progreso: 154.5 MB / 154.5 MB (100.0%)
  [✓] Descarga completada: K002.rar
  Descomprimiendo K002.rar...
  [✓] Descomprimido exitosamente mediante 'tar' del sistema.

Descargando KA01.rar desde Zenodo...
  [⬇] Progreso: 158.9 MB / 158.9 MB (100.0%)
  [✓] Descarga completada: KA01.rar
  Descomprimiendo KA01.rar...
  [✓] Descomprimido exitosamente mediante 'tar' del sistema.

Descargando KI01.rar desde Zenodo...
  [⬇] Progreso: 167.1 MB / 167.1 MB (100.0%)
  [✓] Descarga completada: KI01.rar
  Descomprimiendo KI01.rar...
  [✓] Descomprimido exitosamente mediante 'tar' del sistema.

Descargando KB23.rar desde Zenodo...
  [⬇] Progreso: 163.0 MB / 163.0 MB (100.0%)
  [✓] Descarga completada: KB23.r

# 📘 Guía de Nomenclatura del Dataset de Paderborn

---

## 1. Prefijo de Estado del Rodamiento (Archivos `.rar`)

| Sigla | Significado en Alemán | Descripción en Español | Tipo de Etiqueta (`Label`) |
| :--- | :--- | :--- | :--- |
| **`K0` / `K00x`** | *Kontrolliert* | **Sano / Normal** (Sin desgaste previo) | `Label 0` |
| **`KA`** | *Kugellager Außenring* | **Falla en Pista Exterior** (*Outer Race Fault*) | `Label 1` |
| **`KI`** | *Kugellager Innenring* | **Falla en Pista Interior** (*Inner Race Fault*) | `Label 1` |
| **`KB`** | *Kombinierter Schaden* | **Falla Combinada** (Múltiples elementos dañados) | `Label 1` |

---

## 2. Clasificación por Método de Daño

### A. Fallas Artificiales (Mecanizadas por EDM / Incisión)
Daños provocados intencionalmente mediante descargas eléctricas o grabado antes del montaje:
* **Pista Exterior (KA):** `KA01.rar`, `KA03.rar`, `KA05.rar`, `KA07.rar`, `KA08.rar`, `KA09.rar`
* **Pista Interior (KI):** `KI01.rar`, `KI03.rar`, `KI05.rar`, `KI07.rar`, `KI08.rar`, `KI09.rar`

### B. Fallas Reales / Naturales (Fatiga / Vida Acelerada)
Daños generados por desprendimiento de material (*pitting*) tras operación continua bajo carga:
* **Pista Exterior (KA):** `KA04.rar`, `KA15.rar`, `KA16.rar`, `KA22.rar`, `KA30.rar`
* **Pista Interior (KI):** `KI04.rar`, `KI14.rar`, `KI16.rar`, `KI18.rar`, `KI21.rar`
* **Combinadas (KB):** `KB23.rar`, `KB24.rar`, `KB27.rar`

---

## 3. Parámetros Operativos (Nomenclatura interna de los `.mat`)

* **`N` (Velocidad de Rotación):** `N09` = 900 RPM (15 Hz) \| `N15` = 1500 RPM (25 Hz)
* **`M` (Torque / Par Motor):** `M01` = 0.1 Nm \| `M07` = 0.7 Nm
* **`F` (Fuerza Radial):** `F04` = 400 N \| `F10` = 1000 N

> **Ejemplo:** `N15_M07_F10_K001_1.mat` $\rightarrow$ Rodamiento sano **K001** a **1500 RPM**, **0.7 Nm** de torque y **1000 N** de fuerza radial.

---

## 4. Ejemplos para la Lista `ARCHIVOS_A_DESCARGAR`

```python
# Opción 1: Solo Motores Sanos (Entrenamiento exclusivo del Quantum Autoencoder)
ARCHIVOS_A_DESCARGAR = ["K001.rar", "K002.rar", "K003.rar", "K004.rar", "K005.rar", "K006.rar"]

# Opción 2: Muestra Balanceada Rápida (Sano + 3 tipos de fallas)
ARCHIVOS_A_DESCARGAR = ["K001.rar", "KA01.rar", "KI01.rar", "KB23.rar"]

# Opción 3: Solo Fallas Reales por Fatiga
ARCHIVOS_A_DESCARGAR = ["K001.rar", "K002.rar", "KA04.rar", "KI04.rar", "KB23.rar"]